# CLIP embeddings for Lab 1 images (Colab T4 GPU)

**Что делает:** вычисляет CLIP ViT-B/32 эмбеддинги для ~16 349 картинок размеченных постов, сохраняет parquet 16 349 × 513 (author, post, 512-dim вектор).

**Время на T4:** ~2-5 минут.

**Что нужно загрузить в Google Drive (в папку `kairos`):**
1. `lab1_images_labeled.tar` (~4 ГБ) — архив картинок
2. `image_manifest_subset.csv` (~1 МБ) — маппинг post → file

**Результат вернётся в Drive:** `cache/image_embeddings_clip.parquet`.

## 1. Монтирование Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# После этого увидишь свой Drive в /content/drive/MyDrive/
# Файлы должны лежать в /content/drive/MyDrive/kairos/

import os
DRIVE = '/content/drive/MyDrive/kairos'
assert os.path.exists(f'{DRIVE}/lab1_images_labeled.tar'), f'нет файла в {DRIVE}'
assert os.path.exists(f'{DRIVE}/image_manifest_subset.csv'), 'нет манифеста'
print('OK — файлы на месте')

## 2. Распаковка на локальный диск Colab

Распаковываем из Drive в быстрый локальный диск Colab (`/content/images/`), иначе чтение файлов из Drive будет медленным.

In [ ]:
import tarfile, os, time
os.makedirs('/content/images', exist_ok=True)
t0 = time.time()
with tarfile.open(f'{DRIVE}/lab1_images_labeled.tar') as tar:
    tar.extractall('/content/images/')
print(f'распаковано за {time.time()-t0:.0f}s')
!du -sh /content/images
!ls /content/images | head -5

## 3. Проверяем GPU

In [ ]:
!nvidia-smi
import torch
print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0))

## 4. Устанавливаем open_clip и загружаем модель

In [ ]:
!pip install -q open_clip_torch pandas pyarrow

In [ ]:
import open_clip
import torch

DEVICE = 'cuda'
MODEL_NAME = 'ViT-B-32'
PRETRAINED = 'laion2b_s34b_b79k'  # strong open checkpoint

model, _, preprocess = open_clip.create_model_and_transforms(
    MODEL_NAME, pretrained=PRETRAINED)
model = model.to(DEVICE).eval()
print(f'loaded {MODEL_NAME} ({PRETRAINED})')

## 5. Инференс по картинкам

Пайплайн: читаем манифест → загружаем картинку → preprocess → GPU-батч → CLIP → L2-норм → cpu numpy.

In [ ]:
import pandas as pd
from PIL import Image
import numpy as np
from tqdm.auto import tqdm
import time

mf = pd.read_csv(f'{DRIVE}/image_manifest_subset.csv')
print(f'posts to embed: {len(mf):,}')

BATCH = 128
DIM = 512
all_emb = np.zeros((len(mf), DIM), dtype=np.float32)
ok_mask = np.zeros(len(mf), dtype=bool)

t0 = time.time()
with torch.inference_mode():
    for start in tqdm(range(0, len(mf), BATCH)):
        batch_rows = mf.iloc[start:start+BATCH]
        imgs = []
        idxs = []
        for i, (ri, row) in enumerate(batch_rows.iterrows()):
            path = f'/content/images/{row["file"]}'
            try:
                img = Image.open(path).convert('RGB')
                imgs.append(preprocess(img))
                idxs.append(start + i)
            except Exception as e:
                continue
        if not imgs:
            continue
        batch = torch.stack(imgs).to(DEVICE)
        emb = model.encode_image(batch)
        emb = torch.nn.functional.normalize(emb, p=2, dim=1).cpu().numpy()
        for j, idx in enumerate(idxs):
            all_emb[idx] = emb[j]
            ok_mask[idx] = True

print(f'\ndone in {time.time()-t0:.0f}s. ok: {ok_mask.sum():,} / {len(mf):,}')

## 6. Сохранение в parquet и выгрузка в Drive

In [ ]:
cols = [f'c{i:03d}' for i in range(DIM)]
df = pd.DataFrame(all_emb, columns=cols)
df.insert(0, 'post', mf['post'].values)
df.insert(0, 'author', mf['author'].values)
df['_ok'] = ok_mask

out_path = f'{DRIVE}/image_embeddings_clip.parquet'
df.to_parquet(out_path, index=False)
print(f'wrote {out_path}  ({len(df):,} rows × {df.shape[1]} cols)')

## Готово

Скачай `image_embeddings_clip.parquet` из Drive в `cache/` локально:
```bash
cp /path/to/Drive/kairos/image_embeddings_clip.parquet cache/
```

Или через Drive UI — скачай файл, положи в `kairos-dataset/cache/`.